# Power balance analysis example

This notebook demonstrates the tools in `power_balance.py` on a Turner-benchmark
capacitive RF discharge (450 V, 30 mTorr He, 13.56 MHz -- the parameters in this
directory's `Turner_inputs.py`), whose diagnostics were collected into a
directory named `diags_Turner`.

It's a template: run it once your own `diags_Turner` run has produced output
(`diagnostic_times.dat`, `interval_*`, `time_averaged_*`). Every cell below has
been validated against an equivalent real dataset with the same discharge
parameters, so the only thing that should need to change for a different case
is the `directory`, `voltage`, and species names in the second cell.

In [ ]:
import json

import numpy as np
import matplotlib.pyplot as plt

from analysis import Analysis
import power_balance as pb

## 1. Load the diagnostics and define the drive

`power_balance_components` needs to know the applied wall voltage (as a
function of time in seconds) and the drive frequency to compute `P_wall`
(the I*V power measured at the biased wall) -- both are read straight out of
this repo's own `Turner_inputs.py` parameters. Species default to
`electrons`/`He`, matching this benchmark; pass `electrons=`/`ions=` explicitly
if yours differ.

In [ ]:
diag = Analysis('diags_Turner', quiet_startup=True)

rf_freq = 13.56e6  # Hz
voltage_rf = 450.0  # V
voltage = lambda t: voltage_rf * np.sin(2 * np.pi * rf_freq * t)

## 2. Full power balance breakdown

`power_balance_components` auto-detects which channels are present -- here,
a purely capacitive, wall-biased discharge -- and returns a flat dict with
everything needed for the rest of this notebook (and easy to serialize to JSON).

In [ ]:
components = pb.power_balance_components(diag, voltage=voltage, rf_freq=rf_freq)

print('-'*60)
print(f"Power balance ({diag.directory.split('/')[-1]})".center(60))
print('-'*60)
print(f"  P_wall (I*V)                     = {components['P_wall']:8.2f}   W/m^2")
print(f"  P_in   (capacitive, e + ion)      = {components['P_in']:8.2f}   W/m^2")
print(f"    electrons                       = {components['P_in_e']:8.2f}   W/m^2")
print(f"    ions                            = {components['P_in_i']:8.2f}   W/m^2")
print(f"  P_loss (collisions + wall flux)   = {components['P_loss']:8.2f}   W/m^2")
print(f"    collisions                      = {components['P_coll_e'] + components['P_coll_i']:8.2f}   W/m^2")
print(f"    wall flux                       = {components.get('P_wallflux_e', 0) + components.get('P_wallflux_i', 0):8.2f}   W/m^2")

## 3. Visualizing the balance

`compare_power_balance`/`plot_power_balance_comparison` are built for
comparing several simulations side by side (e.g. a dt-convergence sweep --
see section 5), but work just as well as a single-case summary: one bar for
`P_wall`, one for `P_in` (split by species, and by capacitive/inductive
mechanism where both are present), one for `P_out`. The rendering is chosen
automatically -- the detailed per-species view here, since nothing is
negative; `plot_power_balance_by_mechanism` kicks in automatically instead
if a component ever nets negative (e.g. capacitive power flowing back into
the field at a coarse timestep). The percent label is the self-consistency
check `(P_in - P_out)/P_in`.

In [ ]:
components['label'] = diag.directory.split('/')[-1]

fig, ax = pb.plot_power_balance_comparison([components], figsize=(4.5, 6))
plt.show()

## 4. Where does the power go? (sources and sinks per species)

`plot_species_sources_sinks` breaks the same numbers down differently: one
panel per species, with each mechanism as a bar extending right (net gain)
or left (net loss) from zero. This is the view to reach for when a species'
capacitive or inductive term nets negative -- the aggregate view above would
otherwise just show a smaller total without explaining why.

In [ ]:
fig, axes = pb.plot_species_sources_sinks(components)
plt.show()

## 5. Saving results

Handy for tabulating across runs, or reloading later without recomputing.

In [ ]:
with open('power_balance_results.json', 'w') as f:
    json.dump(components, f, indent=2, default=float)

print('Saved power balance results to power_balance_results.json')

## 6. Comparing across multiple runs (e.g. a dt sweep)

Everything above also works with a *list* of cases, which is where
`compare_power_balance` and `plot_power_balance_convergence` earn their keep
-- e.g. comparing `diags_Turner` runs at several timesteps to check
convergence. That needs more than one directory to be meaningful, so it's
left here as a pattern to copy rather than a cell to run against just
`diags_Turner`:

```python
sweep_dirs = ['diags_Turner_dt1', 'diags_Turner_dt2', 'diags_Turner_dt4', 'diags_Turner_dt8']

cases = [
    dict(directory=d, label=d.replace('diags_Turner_', ''), voltage=voltage, rf_freq=rf_freq)
    for d in sweep_dirs
]

# Grouped bar chart across all cases (same auto-dispatch as section 3):
fig, ax, results = pb.compare_power_balance(cases, figsize=(9, 6))
plt.show()

# Convergence trend only, x-axis in the order `cases` was given:
fig, ax = pb.plot_power_balance_convergence(results)
plt.show()
```